In [17]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# xx_monthly_users.py
# Purpose of Script: Process Table of Monthly Users (Sourced from External
# Reports & Saved in Excel).
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initialization ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Google Drive
#~~~~~~~~~~~~~~~~~~~~~~~~~~
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Libraries
#~~~~~~~~~~~~~~~~~~~~~~~~~~
import numpy as np
import pandas as pd

In [19]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~
# Define Input/Output Paths
#~~~~~~~~~~~~~~~~~~~~~~~~~~
path_users = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/01_final_data/04_supporting/"
path_out = "/content/drive/MyDrive/Colab Notebooks/hsds/xx_dissertation/03_outputs/"

In [20]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Import Data ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df = pd.read_excel(f"{path_users}xx_average_monthly_users.xlsx")

In [21]:
# Expand to form daily data
df_exp = df.copy()
df_exp["start_date"] = pd.to_datetime(df_exp["start_date"])
df_exp["end_date"] = pd.to_datetime(df_exp["end_date"])

# Find Min and Max Dates
min_start = df_exp["start_date"].min()
max_end = df_exp["end_date"].max()
sample_range = pd.date_range(min_start, max_end, freq="D")

# Isolate Platforms
plats = df_exp["platform"].unique()

# Build Cross Table
df_users = pd.MultiIndex.from_product([plats, sample_range], names=["platform","date"]).to_frame(index=False)

# Filter Data Table
df_cut = df[df["country"] == "all"]
df_cut = df_cut[["platform","start_date","end_date","users"]]

# Join Original Data Table
df_users = df_users.merge(df_cut, on="platform", how="left")

# Filter User Data for Correct Date Range
df_users = df_users[(df_users["date"] >= df_users["start_date"]) &
                    (df_users["date"] <= df_users["end_date"])]

df_users = df_users.drop(columns= ["start_date","end_date"])
df_users = df_users[df_users["date"] >= "2025-01-01"]
df_users = df_users.reset_index()

In [22]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Export Data ----
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_users.to_csv(f"{path_out}00_average_monthly_users_clean.csv")